# Updraft sondes — `w_air` either side of cutdown

For every flight in `cutdown_reference_20260914.csv` (hailsondes that separated cleanly from
the balloon), write a copy of its v2026 `.alt.csv` with three extra columns, all in m/s and
**positive up**, like Sparv's barometric `Rise speed`:

- `w_buoy` — still-air ascent rate of the balloon flight train; filled before cutdown only.
- `V_T` — still-air terminal velocity of the detached sonde, negative (falling); filled from
  cutdown + `GAP_S` only.
- `w_air` — the implied vertical air motion, `Rise speed` minus whichever of the two is filled.

| leg | rows | method | `w_buoy` | `V_T` | `w_air` |
|---|---|---|---|---|---|
| balloon attached | before cutdown | `wbuoy_drag_short.ipynb` §5 | filled | blank | `rise − w_buoy` |
| settling | cutdown to cutdown + `GAP_S` | — | blank | blank | blank |
| detached | from cutdown + `GAP_S` | `vt_detached_short.ipynb` §3 | blank | filled | `rise − V_T` |

`w_buoy(ρ) = v_T(ρ₀)·(ρ₀/ρ)^(1/6)`, from the net free lift of balloon plus payload at the
case's launch density (Marinescu et al. 2020, *Mon. Wea. Rev.* 148, Eq. B1), at
`C_D = 0.35`. `|V_T|(ρ) = sqrt(2mg / (ρ C_D A))` is the terminal fall speed of the bare sonde as
a smooth sphere, `C_D = 0.47`, recorded with a negative sign because the sonde falls.

**Settling gap.** After cutdown the sonde has to accelerate from moving with the balloon to
terminal fall. A sphere dropped from rest under quadratic drag follows
`v = V_T tanh(g t / V_T)`, reaching 99 % of `V_T` at `t = atanh(0.99)·V_T/g`. At the cutdown
densities in this set (0.31–0.76 kg/m³, `V_T` 21–37 m/s) that is 6–10 s. The barometric rise
speed takes longer, typically levelling off 10–20 s after the logged cutdown time, because of
the retrieval's smoothing and uncertainty in the logged time. `GAP_S = 20` s covers both;
§2 prints the theoretical settling time per flight as a check.

**Caveats** carried over from the source notebooks: `w_buoy` has no spring-scale free lift
behind it, so its level is uncertain by about ±1 m/s (`C_D` window plus fill), and `V_T` is for
a clean sonde, so where ice has accumulated `w_air` overstates the downdraft. No rows are masked
beyond the settling gap: pre-launch and post-landing rows carry values that are not
physically meaningful.

**Input:** `cutdown_reference_20260914.csv`, `v2026/<yyyymmdd>/<category>/<stem>.alt.csv`.
**Output:**
- `/home/meso/data/icechip-hailsonde-data/v2026_w_air/csv/<stem>.w_air.csv` — the original file (both `#` header lines and all
  columns, values untouched) plus `w_buoy`, `V_T` and `w_air`.
- `/home/meso/data/icechip-hailsonde-data/v2026_w_air/png/<stem>.w_air.png` — a time series of all four per flight (§4).

Joshua Soderholm - 14 September 2026

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ============================== PARAMETERS ==================================
GAP_S = 20.0    # s after cutdown during which the sonde is still settling; w_air is NaN there

REFERENCE = Path("cutdown_reference_20260914.csv")                # flights that cut down cleanly
DATA      = Path("/home/meso/data/icechip-hailsonde-data/v2026")  # campaign root: <yyyymmdd>/<category>/<stem>.alt.csv
OUT_DIR   = Path("/home/meso/data/icechip-hailsonde-data/v2026_w_air")  # output root
CSV_DIR   = OUT_DIR / "csv"                                       # one <stem>.w_air.csv per flight
PNG_DIR   = OUT_DIR / "png"                                       # one <stem>.w_air.png per flight

# hardware versions, identified by the sonde id in the filename stem
V2_ID_MIN, V2_ID_MAX = 15312, 15399           # v1: id < V2_ID_MIN;  v2: V2_ID_MIN..V2_ID_MAX
PAYLOAD  = {"v1": 0.025, "v2": 0.020}         # kg, sonde mass (clean, no ice)
DIAMETER = {"v1": 0.065, "v2": 0.050}         # m, sonde sphere diameter

# balloon-attached flight train (wbuoy_drag_short.ipynb)
M_BALLOON          = 0.020    # kg, balloon mass
D_LAUNCH           = 0.55     # m, balloon diameter at launch
CD_BALLOON         = 0.35     # dimensionless, central balloon drag coefficient (window 0.2-0.5)
LAUNCH_AGL         = 100.0    # m AGL, band used to fix the launch density
MIN_LAUNCH_SAMPLES = 5        # fewer than this in the band -> use the lowest samples instead

# detached sonde (vt_detached_short.ipynb)
CD_SPHERE = 0.47    # dimensionless, smooth-sphere drag coefficient, subcritical Re
# ============================================================================

# physical constants
G       = 9.80665             # m/s2, standard gravity
R_DRY   = 287.05              # J/kg/K, gas constant for dry air
M_RATIO = 4.0026 / 28.9647    # helium / dry air molar mass -> density ratio
C_TO_K  = 273.15              # K, offset from degrees Celsius to kelvin

# balloon geometry at launch, treated as a sphere of diameter D_LAUNCH
V_LAUNCH = (4.0 / 3.0) * np.pi * (D_LAUNCH / 2.0) ** 3    # m3
A_LAUNCH = np.pi * (D_LAUNCH / 2.0) ** 2                  # m2

print(f"settling gap after cutdown: {GAP_S:.0f} s")

## 1. Physics

The balloon functions are those of `wbuoy_drag_short.ipynb` §4–5 and the sphere function that
of `vt_detached_short.ipynb`, unchanged.

In [2]:
def net_free_lift(rho_launch, m_payload, V=V_LAUNCH):
    """
    Net free lift of the flight train, evaluated at launch.

      L = V * rho * (1 - M_He / M_air) - m_balloon - m_payload

    The helium expands with falling pressure so V * rho, and hence L, is
    constant for the rest of the ascent.

    Reference
    ---------
    Marinescu et al. (2020), Mon. Wea. Rev. 148, 4435-4452, Appendix B.

    Parameters
    ----------
    rho_launch : float or array_like
        Air density at launch (kg m⁻³).
    m_payload : float or array_like
        Payload mass (kg).
    V : float
        Balloon volume at launch (m³).

    Returns
    -------
    lift : float or array_like
        Net free lift (kg).
    """
    return V * rho_launch * (1.0 - M_RATIO) - M_BALLOON - m_payload


def area_at(rho, rho_launch):
    """
    Balloon cross-section after free expansion.

      A = A_launch * (rho_launch / rho)^(2/3)      since V ~ 1/rho and A ~ V^(2/3)

    Parameters
    ----------
    rho : float or array_like
        Ambient air density (kg m⁻³).
    rho_launch : float or array_like
        Air density at launch (kg m⁻³).

    Returns
    -------
    area : float or array_like
        Cross-sectional area (m²).
    """
    return A_LAUNCH * (rho_launch / rho) ** (2.0 / 3.0)


def v_terminal(rho, cd, lift, A):
    """
    Terminal ascent rate of the balloon, Marinescu et al. (2020) Eq. (B1).

      v_T = sqrt(2 g L / (rho C_D A))

    Reference
    ---------
    Marinescu et al. (2020), Mon. Wea. Rev. 148, 4435-4452, Appendix B.

    Parameters
    ----------
    rho : float or array_like
        Ambient air density (kg m⁻³).
    cd : float
        Drag coefficient (dimensionless).
    lift : float or array_like
        Net free lift (kg).  Non-positive values return 0.
    A : float or array_like
        Balloon cross-section (m²).

    Returns
    -------
    v_t : float or array_like
        Terminal ascent rate (m s⁻¹).
    """
    lift = np.clip(lift, 0.0, None)   # no net lift means no ascent: bound at zero rather than NaN
    return np.sqrt(2.0 * G * lift / (rho * cd * A))


def sphere_vt(rho, mass, diameter, cd=CD_SPHERE):
    """
    Terminal velocity of a rigid sphere falling in still air, where weight balances drag.

      V_T = sqrt(2 m g / (rho * C_D * A)),   A = pi D^2 / 4

    Parameters
    ----------
    rho : float or array_like
        Air density (kg m⁻³).
    mass : float
        Sphere mass (kg).
    diameter : float
        Sphere diameter (m).
    cd : float
        Drag coefficient (dimensionless).

    Returns
    -------
    vt : float or array_like
        Terminal velocity (m s⁻¹), positive number, same shape as `rho`.
    """
    area   = np.pi * diameter ** 2 / 4.0    # m2, frontal area
    weight = mass * G                       # N
    return np.sqrt(2.0 * weight / (rho * cd * area))

## 2. Flights and cutdown times

One row per flight in the reference sheet. Each is matched to its v2026 file on the sonde id,
the third underscore-separated field of the stem, and the match must be unique.

The files carry `UTC time` as `hh:mm:ss` with no date, and several flights cross midnight UTC.
Time since cutdown is therefore taken as the difference in seconds of day wrapped into
±12 h, which is unambiguous because no flight lasts that long.

In [3]:
def _num(column):
    """Parse a text column to float; blank or unparseable entries become NaN."""
    stripped = column.astype(str).str.strip()
    blanked  = stripped.replace("", np.nan)
    return pd.to_numeric(blanked, errors="coerce")


def sonde_id(stem):
    """Sonde serial number: the third underscore-separated field of the file stem."""
    return int(stem.split("_")[2])


def sonde_version(sid):
    """Hardware version from the serial range: "v1" or "v2"."""
    if sid < V2_ID_MIN:
        return "v1"
    if sid <= V2_ID_MAX:
        return "v2"
    raise ValueError(f"sonde id {sid} is outside the known ranges "
                     f"(v1 < {V2_ID_MIN}, v2 {V2_ID_MIN}-{V2_ID_MAX})")


def find_file(sid, root=DATA):
    """The single .alt.csv under `root` whose stem carries sonde id `sid`."""
    hits = [f for f in sorted(root.rglob("*.alt.csv")) if sonde_id(f.name[: -len(".alt.csv")]) == sid]
    if len(hits) != 1:
        raise RuntimeError(f"expected one .alt.csv for sonde {sid}, found {len(hits)}: "
                           f"{[f.name for f in hits]}")
    return hits[0]


def seconds_since(utc_time, event_hms):
    """
    Seconds from an event to each sample, from time-of-day strings only.

    Parameters
    ----------
    utc_time : Series of str
        Sample times, hh:mm:ss (UTC), no date.
    event_hms : str
        Event time, hh:mm:ss (UTC).

    Returns
    -------
    dt : Series of float
        Seconds after the event (negative before), wrapped into [-43200, 43200)
        so a flight crossing midnight UTC stays continuous.
    """
    sample_sec = pd.to_timedelta(utc_time.str.strip()).dt.total_seconds()
    event_sec  = pd.to_timedelta(event_hms.strip()).total_seconds()
    HALF_DAY, DAY = 43200.0, 86400.0    # s
    return (sample_sec - event_sec + HALF_DAY) % DAY - HALF_DAY


# read the reference sheet; one row per flight
flights = pd.read_csv(REFERENCE, skipinitialspace=True)
flights.columns = [c.strip() for c in flights.columns]

# attach each flight's file, version and cutdown density / settling time
# list (dim: flight) of dict
rows = []
for _, flight in flights.iterrows():
    sid      = int(flight["Sonde ID"])
    csv_path = find_file(sid)
    version  = sonde_version(sid)

    # sample nearest the logged cutdown time
    raw = pd.read_csv(csv_path, skiprows=2, skipinitialspace=True, dtype=str, keep_default_na=False)
    raw.columns = [c.strip() for c in raw.columns]
    dt = seconds_since(raw["UTC time"], flight["cutdown time (UTC+0)"])
    cut_idx = dt.abs().idxmin()

    # density there, the sonde's V_T at that density, and the time to reach 99 % of it from rest
    p_cut   = _num(raw["Pressure (Pa)"])[cut_idx]                 # Pa
    t_cut   = _num(raw["Temperature (C)"])[cut_idx] + C_TO_K      # K
    rho_cut = p_cut / (R_DRY * t_cut)                             # kg/m3
    vt_cut  = sphere_vt(rho_cut, PAYLOAD[version], DIAMETER[version])   # m/s
    t99     = np.arctanh(0.99) * vt_cut / G                       # s, from v = V_T tanh(g t / V_T)

    rows.append({
        "sonde_id":   sid,
        "stem":       csv_path.name[: -len(".alt.csv")],
        "version":    version,
        "nearest_s":  dt[cut_idx],                                # s, cutdown to nearest sample
        "alt_cut":    _num(raw["Altitude (m MSL)"])[cut_idx],     # m MSL
        "rho_cut":    rho_cut,
        "vt_cut":     vt_cut,
        "t99_s":      t99,
    })

# summary table (dim: flight)
matched = pd.DataFrame(rows)
print(f"{len(matched)} flights matched to files; settling gap {GAP_S:.0f} s, "
      f"theoretical 99 % settling {matched.t99_s.min():.1f}-{matched.t99_s.max():.1f} s\n")
print(matched.to_string(index=False, float_format=lambda v: f"{v:8.2f}"))

25 flights matched to files; settling gap 20 s, theoretical 99 % settling 5.7-10.1 s

 sonde_id                         stem version  nearest_s  alt_cut  rho_cut   vt_cut    t99_s
    15312        2025-05-18_1846_15312      v2       1.00     7664     0.51    28.76     7.76
     9516         2025-05-18_2315_9516      v1       0.00     5813     0.61    22.64     6.11
    15297 2025-05-18_2314_15297_merged      v1       0.00     5449     0.66    21.85     5.90
    15272        2025-05-23_2226_15272      v1       0.00     5569     0.65    21.98     5.93
    15294        2025-05-23_2235_15294      v1       0.00     8541     0.48    25.67     6.93
    15302        2025-05-23_2240_15302      v1       0.00     7900     0.50    24.98     6.74
    15299        2025-05-25_1923_15299      v1       0.00     8151     0.49    25.40     6.86
    15301        2025-05-25_1927_15301      v1       0.00     6727     0.56    23.72     6.40
     9518         2025-05-25_1956_9518      v1      -1.00     3468  

## 3. `w_buoy`, `V_T` and `w_air` for every row, and write

Every row of the original file is kept, including repeated timestamps and pre-launch samples,
and its text is written back unchanged; only the three new columns are added, rounded to
0.01 m/s. Density is dry-air `p / (R_d T)` throughout, as in both source notebooks.

The launch density `ρ₀` is the median over the case's samples at or below `LAUNCH_AGL`, or its
lowest `MIN_LAUNCH_SAMPLES` samples when it has too few there — as in `wbuoy_drag_short.ipynb`.

In [ ]:
def launch_density(agl, rho, launch_agl=LAUNCH_AGL, min_samples=MIN_LAUNCH_SAMPLES):
    """
    Air density at launch for one case, from the lowest samples it has.

    No extrapolation to the ground: a case whose telemetry starts high gets a low value.

    Parameters
    ----------
    agl : Series
        Height of each sample (m AGL).
    rho : Series
        Air density of each sample (kg m⁻³), same index as `agl`.
    launch_agl : float
        Samples at or below this height (m AGL) define launch.
    min_samples : int
        If fewer samples lie below `launch_agl`, the lowest `min_samples` are used.

    Returns
    -------
    rho0 : float
        Median density over those samples (kg m⁻³).
    """
    usable = agl.notna() & rho.notna()
    low    = usable & (agl <= launch_agl)
    if low.sum() < min_samples:
        lowest_idx = agl[usable].nsmallest(min_samples).index
        return rho[lowest_idx].median()
    return rho[low].median()


def main(rise, p, temp, agl, dt_cut, version, gap_s=GAP_S):
    """
    Still-air sonde velocity and vertical air motion either side of cutdown for one flight.

      before cutdown:          w_air = rise - w_buoy(rho)
      cutdown .. + gap_s:      w_air = NaN
      after cutdown + gap_s:   w_air = rise - V_T(rho)

    with w_buoy(rho) = v_T(rho0) (rho0/rho)^(1/6) (Marinescu et al. 2020, Eq. B1)
    and V_T(rho) = -sqrt(2 m g / (rho C_D pi D^2/4)) for the bare sonde, negative
    because it falls.

    Parameters
    ----------
    rise : Series
        Barometric rise speed (m s⁻¹), positive up.
    p : Series
        Pressure (Pa).
    temp : Series
        Temperature (°C).
    agl : Series
        Height (m AGL).
    dt_cut : Series
        Time after cutdown (s), negative before.
    version : str
        Sonde hardware version, "v1" or "v2".
    gap_s : float
        Settling time after cutdown (s) over which all outputs are left NaN.

    Returns
    -------
    w_buoy : Series
        Still-air ascent rate of the balloon flight train (m s⁻¹), positive up;
        NaN from cutdown onward.
    v_t : Series
        Still-air terminal velocity of the detached sonde (m s⁻¹), positive up so
        negative; NaN before cutdown + gap_s.
    w_air : Series
        Vertical air motion (m s⁻¹), positive up; NaN inside the settling gap.
    """
    # check inputs before any computation
    if version not in PAYLOAD:
        raise ValueError(f"version must be one of {list(PAYLOAD)}, found {version!r}")
    if gap_s < 0.0:
        raise ValueError(f"gap_s must be non-negative, found {gap_s}")
    if not (dt_cut < 0.0).any() or not (dt_cut >= gap_s).any():
        raise RuntimeError(f"cutdown must fall inside the record with {gap_s:.0f} s to spare; "
                           f"record spans {dt_cut.min():.0f} to {dt_cut.max():.0f} s from cutdown")

    # dry-air density (kg/m3)
    rho = p / (R_DRY * (temp + C_TO_K))

    # balloon attached: buoyant ascent rate (m/s, positive up) of this flight train at each row's density
    rho0     = launch_density(agl, rho)                        # kg/m3
    lift     = net_free_lift(rho0, PAYLOAD[version])           # kg
    w_buoy_all = v_terminal(rho, CD_BALLOON, lift, area_at(rho, rho0))

    # detached: terminal velocity (m/s, positive up, so negative) of the bare sonde at each row's density
    v_t_all = -sphere_vt(rho, PAYLOAD[version], DIAMETER[version])

    # keep each only on the leg where it applies; the settling gap matches neither and stays NaN
    before_cut   = dt_cut < 0.0
    after_settle = dt_cut >= gap_s
    w_buoy = w_buoy_all.where(before_cut)
    v_t    = v_t_all.where(after_settle)

    # air motion (m/s, positive up): observed minus still-air velocity on whichever leg is filled
    w_air = pd.Series(np.nan, index=rise.index)
    w_air[before_cut]   = rise[before_cut] - w_buoy[before_cut]
    w_air[after_settle] = rise[after_settle] - v_t[after_settle]
    return w_buoy, v_t, w_air


def _fmt_velocity(series):
    """Two-decimal text for the output file; NaN becomes a blank field, as in the original."""
    return series.map(lambda v: "" if np.isnan(v) else f"{v:.2f}")


CSV_DIR.mkdir(parents=True, exist_ok=True)

# one output file per flight
# list (dim: flight) of dict, summary of what was written
written = []
for _, flight in flights.iterrows():
    sid      = int(flight["Sonde ID"])
    csv_path = find_file(sid)
    stem     = csv_path.name[: -len(".alt.csv")]

    # keep the two '#' header lines and every field as text, so they are written back unchanged;
    # newline="" keeps the file's own line ending (the Windsond files use \r\n)
    with open(csv_path, newline="") as fh:
        header_lines = [fh.readline(), fh.readline()]
    line_end = "\r\n" if header_lines[0].endswith("\r\n") else "\n"
    raw = pd.read_csv(csv_path, skiprows=2, dtype=str, keep_default_na=False)
    names = [c.strip() for c in raw.columns]     # stripped names, for lookup only

    # unpack the fields main needs, as floats
    fields = dict(zip(names, raw.columns))
    rise   = _num(raw[fields["Rise speed (m/s)"]])          # m/s, positive up
    p      = _num(raw[fields["Pressure (Pa)"]])             # Pa
    temp   = _num(raw[fields["Temperature (C)"]])           # deg C
    agl    = _num(raw[fields["Height (m AGL)"]])            # m AGL
    dt_cut = seconds_since(raw[fields["UTC time"]], flight["cutdown time (UTC+0)"])   # s

    # run retrieval
    w_buoy, v_t, w_air = main(rise, p, temp, agl, dt_cut, sonde_version(sid), gap_s=GAP_S)

    # append the new columns in the file's own ", " style and write
    out = raw.copy()
    out[" w_buoy (m/s)"] = _fmt_velocity(w_buoy)
    out[" V_T (m/s)"]    = _fmt_velocity(v_t)
    out[" w_air (m/s)"]  = _fmt_velocity(w_air)
    out_path = CSV_DIR / f"{stem}.w_air.csv"
    with open(out_path, "w", newline="") as fh:
        fh.writelines(header_lines)
        out.to_csv(fh, index=False, lineterminator=line_end)

    # record coverage of each leg
    written.append({
        "stem":           stem,
        "rows":           len(out),
        "attached":       int((dt_cut < 0.0).sum()),
        "gap":            int(((dt_cut >= 0.0) & (dt_cut < GAP_S)).sum()),
        "detached":       int((dt_cut >= GAP_S).sum()),
        "w_air_up_max":   w_air[dt_cut < 0.0].max(),          # m/s, balloon-attached leg, largest w_air
        "w_air_down_max": w_air[dt_cut >= GAP_S].max(),       # m/s, detached leg, largest w_air
        "w_air_down_min": w_air[dt_cut >= GAP_S].min(),       # m/s, detached leg, smallest (most negative) w_air
    })

# summarise
summary = pd.DataFrame(written)
print(f"wrote {len(summary)} files to {CSV_DIR}/\n")
print(summary.to_string(index=False, float_format=lambda v: f"{v:7.2f}"))

## 4. Time series per case

One figure per flight, saved as `/home/meso/data/icechip-hailsonde-data/v2026_w_air/png/<stem>.w_air.png` and not shown inline. Plotted
against time since cutdown rather than altitude: several sondes kept rising or hovered after
cutdown, so altitude is not monotonic on the detached leg and a profile would fold back on
itself.

Rise speed is observed (blue); `w_buoy` and `V_T` are the still-air references (grey, solid and
dashed), each drawn only on its own leg; `w_air` is the difference (orange). The shaded band
is the settling gap. The whole record is plotted, pre-launch and post-landing included.

In [ ]:
# Palette follows wbuoy_drag_short.ipynb: blue = observed, orange = derived,
# neutral ink = a still-air reference rather than a categorical series.
C_OBS, C_AIR = "#2a78d6", "#eb6834"
INK, INK2, GRID, SURF = "#0b0b0b", "#52514e", "#dcdbd6", "#fcfcfb"
plt.rcParams.update({
    "figure.facecolor": SURF, "axes.facecolor": SURF, "savefig.facecolor": SURF,
    "axes.edgecolor": GRID, "axes.labelcolor": INK2, "text.color": INK,
    "xtick.color": INK2, "ytick.color": INK2, "font.size": 9,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False, "axes.axisbelow": True,
    "legend.frameon": False, "figure.dpi": 130,
})


def plot_case(csv_path, png_path, title, cutdown_hms, gap_s=GAP_S):
    """
    Time series of rise speed, still-air velocity and w_air for one flight, saved to file.

    Parameters
    ----------
    csv_path : Path
        Output file written in §3, <stem>.w_air.csv.
    png_path : Path
        Where the figure is saved.
    title : str
        Figure title.
    cutdown_hms : str
        Cutdown time, hh:mm:ss (UTC).
    gap_s : float
        Settling time after cutdown (s), shaded.
    """
    # read the written file back, so the figure shows exactly what is in it
    case = pd.read_csv(csv_path, skiprows=2, skipinitialspace=True, dtype={"UTC time": str})
    case.columns = [c.strip() for c in case.columns]
    dt_cut = seconds_since(case["UTC time"], cutdown_hms)    # s after cutdown

    # cutdown and settling gap as references
    fig, ax = plt.subplots(figsize=(10.0, 4.8))
    ax.axvspan(0.0, gap_s, color=INK2, alpha=0.12, lw=0, label=f"settling gap ({gap_s:.0f} s)")
    ax.axvline(0.0, color=INK2, lw=1.0, ls="--")
    ax.axhline(0.0, color=INK2, lw=0.8)

    # observed, still-air references (each NaN off its own leg, so the lines break there), derived
    ax.plot(dt_cut, case["Rise speed (m/s)"], color=C_OBS, lw=1.2, label="rise speed (observed)")
    ax.plot(dt_cut, case["w_buoy (m/s)"], color=INK2, lw=1.8, label="$w_{buoy}$ (balloon, still air)")
    ax.plot(dt_cut, case["V_T (m/s)"], color=INK2, lw=1.8, ls=(0, (4, 2)), label="$V_T$ (sonde, still air)")
    ax.plot(dt_cut, case["w_air (m/s)"], color=C_AIR, lw=1.4, label="$w_{air}$")

    # labels, legend, save without showing inline
    ax.set_xlabel("time since cutdown (s)")
    ax.set_ylabel("vertical velocity (m s$^{-1}$), positive up")
    ax.set_title(title, loc="left", fontsize=11, color=INK, pad=8)
    ax.legend(loc="upper right", fontsize=8, ncol=5)
    fig.savefig(png_path, bbox_inches="tight")
    plt.close(fig)


PNG_DIR.mkdir(parents=True, exist_ok=True)

# cutdown altitude per flight, from §2, for the titles
alt_cut_by_sid = matched.set_index("sonde_id").alt_cut    # m MSL, dim: sonde id

# one figure per flight, read from its CSV
for _, flight in flights.iterrows():
    sid         = int(flight["Sonde ID"])
    stem        = find_file(sid).name[: -len(".alt.csv")]
    cutdown_hms = flight["cutdown time (UTC+0)"]
    title = (f"{stem} — {sonde_version(sid)}, cutdown {cutdown_hms} UTC "
             f"at {alt_cut_by_sid[sid] / 1000.0:.1f} km MSL")
    plot_case(
        CSV_DIR / f"{stem}.w_air.csv",
        PNG_DIR / f"{stem}.w_air.png",
        title,
        cutdown_hms,
        gap_s=GAP_S,
    )

print(f"saved {len(flights)} figures to {PNG_DIR}/")